# DATA 201 · Week 3
## Logistic Regression and Classification Foundations

This lab follows the Week 3 slides. We use the **Default** data from ISLP Chapter 4: 10,000 credit-card customers, and we try to tell who is likely to default.

The process we will walk through is the same one as on the slides:

$$
\text{features } X \;\rightarrow\; z \;\rightarrow\; p(X) \;\rightarrow\; \text{threshold} \;\rightarrow\; \text{Yes / No}
$$

A **probability is not a class**. The model first gives a number between 0 and 1. A **threshold** then turns that number into a label.

### How to write answers

After each question you will see an empty box like this:

```python
q1 = """


"""
```

Type your answer between the quotes. Leave it there so you can turn the notebook in with your work.


## Part 1 — Load and inspect the data


Each row is one customer. The columns are:

- `default` — did this customer default? **Yes** or **No**. This is the **target**.
- `student` — is the customer a student?
- `balance` — credit-card balance
- `income` — income

If you already put `Default.csv` in Google Drive, you can change the path below. Otherwise the notebook looks next to this file, then downloads the ISLP Default data.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix

pd.set_option("display.max_rows", 20)
plt.rcParams["figure.figsize"] = (7, 4)

def load_default():
    candidates = [
        Path("Default.csv"),
        Path("/content/drive/MyDrive/Data_201/Datasets/Default.csv"),
    ]
    for path in candidates:
        if path.exists():
            print("Loaded from", path.resolve())
            return pd.read_csv(path)

    url = "https://raw.githubusercontent.com/vincentarelbundock/Rdatasets/master/csv/ISLR/Default.csv"
    print("Loaded Default.csv from the web")
    df = pd.read_csv(url)
    if "rownames" in df.columns:
        df = df.drop(columns=["rownames"])
    return df

Default = load_default()
Default.head()


In [ ]:
Default.shape


In [ ]:
Default.info()


In [ ]:
Default["student"].value_counts()


In [ ]:
Default["default"].value_counts()


In [ ]:
Default[["balance", "income"]].describe()


Look at the tables above, then answer in your own words.


**Q1.** What does each row represent?


In [ ]:
q1 = """


"""


**Q2.** What is the target variable? Is this a regression problem or a classification problem?


In [ ]:
q2 = """


"""


**Q3.** Which columns could be used as predictors (features)?


In [ ]:
q3 = """


"""


**Q4.** Is the target balanced? About how many customers defaulted, compared with how many did not?


In [ ]:
q4 = """


"""


## Part 2 — Why not just use linear regression?

We start with **one predictor**: `balance`. Plot default as 0 / 1 against balance. Then we fit a straight line and a logistic curve on the same picture. This is the idea of ISLP Figure 4.2 from the slides.


In [ ]:
y01 = Default["default"].map({"No": 0, "Yes": 1})

plt.scatter(Default["balance"], y01, alpha=0.15, s=12)
plt.xlabel("Credit-card balance")
plt.ylabel("Default (0 = No, 1 = Yes)")
plt.title("Balance and default")
plt.yticks([0, 1])
plt.show()


In [ ]:
# A straight line can go below 0 or above 1. A probability cannot.
X_bal = Default[["balance"]]
xs = np.linspace(0, Default["balance"].max(), 300).reshape(-1, 1)

lin = LinearRegression().fit(X_bal, y01)
log = LogisticRegression(max_iter=2000).fit(X_bal, y01)
xs_df = pd.DataFrame({"balance": xs.ravel()})

plt.scatter(Default["balance"], y01, alpha=0.12, s=10, label="customers")
plt.plot(xs, lin.predict(xs_df), label="linear regression")
plt.plot(xs, log.predict_proba(xs_df)[:, 1], label="logistic regression")
plt.axhline(0, linestyle="--", linewidth=1)
plt.axhline(1, linestyle="--", linewidth=1)
plt.xlabel("Credit-card balance")
plt.ylabel("Predicted default / probability")
plt.title("Why the linear line is the wrong shape")
plt.legend()
plt.show()


**Q5.** What pattern do you notice between credit-card balance and default?


In [ ]:
q5 = """


"""


**Q6.** Why is ordinary linear regression a poor fit for an outcome that can only be Yes or No? Look at the plot: does the straight line stay between 0 and 1?


In [ ]:
q6 = """


"""


## Part 3 — Prepare the data

Turn the target into 0 / 1, keep `balance` as the only feature for now, and split into **training** and **test** sets.

`stratify=y` keeps about the same mix of Yes / No in both pieces. That matters here because defaults are rare.


In [ ]:
Default["default_num"] = Default["default"].map({"No": 0, "Yes": 1})

X = Default[["balance"]]
y = Default["default_num"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y,
)

print("Training rows:", len(X_train), "  Test rows:", len(X_test))
print("Default rate in training:", y_train.mean().round(4))
print("Default rate in test:    ", y_test.mean().round(4))


**Q7.** Which variable is the predictor, and which is the target?


In [ ]:
q7 = """


"""


**Q8.** Why do we separate the data into a training set and a test set? What would go wrong if we tested the model on the same rows it was trained on?


In [ ]:
q8 = """


"""


**Q9.** Why is `stratify=y` useful when almost all customers have `default = No`?


In [ ]:
q9 = """


"""


## Part 4 — Fit logistic regression using balance

The fitted model has the linear score

$$
z = \beta_0 + \beta_1 X
$$

Then the **logistic (sigmoid) function** turns \(z\) into a probability:

$$
p(X) = \frac{e^{z}}{1+e^{z}} = P(\text{default}=\text{Yes}\mid X)
$$

\(z\) is **not** a probability. \(p(X)\) is.


In [ ]:
logreg = LogisticRegression(max_iter=2000)
logreg.fit(X_train, y_train)

beta_0 = logreg.intercept_[0]
beta_1 = logreg.coef_[0][0]

print("Intercept β0:", round(beta_0, 4))
print("Balance  β1:", round(beta_1, 6))


**Q10.** Is the coefficient for balance positive or negative?


In [ ]:
q10 = """


"""


**Q11.** Based on that sign, what happens to \(z\) as balance increases?


In [ ]:
q11 = """


"""


**Q12.** What should happen to the estimated probability of default as \(z\) increases?


In [ ]:
q12 = """


"""


## Part 5 — From \(z\) to a probability, for one customer

Pick one test customer. Compute \(z\) by hand, then apply the sigmoid. Check that it matches `predict_proba`.


In [ ]:
# A customer near a $1,500 balance is easier to talk about than a near-zero one.
customer = X_test.iloc[[(X_test["balance"] - 1500).abs().argmin()]]
customer


In [ ]:
balance = customer["balance"].iloc[0]
z = beta_0 + beta_1 * balance
p_manual = np.exp(z) / (1 + np.exp(z))

print("Balance:", round(balance, 2))
print("z      :", round(z, 4))
print("p(X)   :", round(p_manual, 4))
print()
print("scikit-learn predict_proba:", logreg.predict_proba(customer))
print("Those two numbers are P(No default) and P(Default). They add to 1.")


**Q13.** Does your hand-calculated \(p(X)\) match the **second** number from `predict_proba`?


In [ ]:
q13 = """


"""


**Q14.** Why does `predict_proba` return two numbers instead of one?


In [ ]:
q14 = """


"""


**Q15.** Is the probability itself the final Yes / No prediction? What else do we still need?


In [ ]:
q15 = """


"""


### By hand, like the lecture (Customer A)

On the slides we used a **made-up** model so the arithmetic stays small:

$$
z = -4 + 1.5\,X_1 - 0.4\,X_2
$$

Customer A: balance \$1,000 so \(X_1=1\) (thousands of dollars), income \$60,000 so \(X_2=6\) (tens of thousands).


In [ ]:
# Fill in the blanks, then run the cell.
X1 = 1
X2 = 6
z_A = -4 + 1.5 * X1 - 0.4 * X2
p_A = np.exp(z_A) / (1 + np.exp(z_A))

print("z for Customer A:", z_A)
print("p for Customer A:", round(p_A, 4))


**Q16.** Using a 0.50 threshold, what class would the lecture model assign to Customer A? Is that customer almost certainly No, or almost certainly Yes?


In [ ]:
q16 = """


"""


## Part 6 — Probability vs. prediction

Now compute \(p(X)\) for every test customer. The usual rule on the slides is:

$$
p(X)\ge 0.50 \Rightarrow \text{Yes (default)}
$$
$$
p(X)< 0.50 \Rightarrow \text{No}
$$

0.50 is **common, not required**. It is a decision rule, not part of the fitted curve.


In [ ]:
prob_default = logreg.predict_proba(X_test)[:, 1]
pred_50 = logreg.predict(X_test)

results = X_test.copy()
results["Actual"] = y_test.values
results["P(Default)"] = prob_default
results["Prediction_0.50"] = pred_50
results.head(15)


In [ ]:
# Customers whose probability is close to the usual cutoff
near_half = results[(results["P(Default)"] > 0.40) & (results["P(Default)"] < 0.60)]
near_half.sort_values("P(Default)")


**Q17.** Find a customer whose predicted probability is close to 0.50. About what is their balance?


In [ ]:
q17 = """


"""


**Q18.** What class does a probability of 0.49 receive at a 0.50 threshold? What about 0.51?


In [ ]:
q18 = """


"""


**Q19.** Are 0.49 and 0.51 very different probabilities? Why can they still produce different class labels?


In [ ]:
q19 = """


"""


## Part 7 — Change the threshold, not the model

Lower the cutoff to 0.30. We do **not** refit the logistic regression. We only change how a probability becomes a label.


In [ ]:
results["Prediction_0.30"] = (results["P(Default)"] >= 0.30).astype(int)

changed = results[results["Prediction_0.50"] != results["Prediction_0.30"]]
print("How many customers changed class:", len(changed))
changed.head(10)


**Q20.** Which kind of customer changed class: high probability, low probability, or the ones in the middle?


In [ ]:
q20 = """


"""


**Q21.** Did their predicted probabilities change when you switched to 0.30?


In [ ]:
q21 = """


"""


**Q22.** Did we retrain the logistic regression model?


In [ ]:
q22 = """


"""


**Q23.** So what actually changed: the fitted curve, or the decision rule?


In [ ]:
q23 = """


"""


## Part 8 — False positives, false negatives

A **confusion matrix** counts the four outcomes. For this data, class 1 means default (Yes).

| | Predicted No | Predicted Yes |
|---|---|---|
| **Actual No** | true negative | false positive (false alarm) |
| **Actual Yes** | false negative (missed default) | true positive |

Lowering the threshold usually catches more real defaults **and** creates more false alarms. That is the tradeoff from the slides.


In [ ]:
def show_confusion(actual, predicted, title):
    cm = confusion_matrix(actual, predicted)
    table = pd.DataFrame(
        cm,
        index=["Actual No", "Actual Yes"],
        columns=["Predicted No", "Predicted Yes"],
    )
    print(title)
    print(table.to_string())
    tn, fp, fn, tp = cm.ravel()
    print(f"false positives (false alarms): {fp}")
    print(f"false negatives (missed defaults): {fn}")
    print()

show_confusion(results["Actual"], results["Prediction_0.50"], "Threshold 0.50")
show_confusion(results["Actual"], results["Prediction_0.30"], "Threshold 0.30")


**Q24.** What happens to the number of **false negatives** when the threshold decreases from 0.50 to 0.30?


In [ ]:
q24 = """


"""


**Q25.** What happens to the number of **false positives**?


In [ ]:
q25 = """


"""


**Q26.** Does lowering the threshold automatically make the model better? Explain.


In [ ]:
q26 = """


"""


**Q27.** The slides ask: would a doctor wait until \(P(\text{disease})>0.50\) before referring a patient? Why might a **lower** threshold be useful in some jobs and a bad idea in others?


In [ ]:
q27 = """


"""


## Part 9 — Two predictors: balance and income

Add `income`. The two features are on very different scales (hundreds vs tens of thousands), so we **split first, then scale**, the same habit as Week 2. Fit the scaler on the training data only.


In [ ]:
X2 = Default[["balance", "income"]]
y = Default["default_num"]

X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2, y, test_size=0.25, random_state=42, stratify=y
)

scaler = StandardScaler()
X2_train_scaled = scaler.fit_transform(X2_train)
X2_test_scaled = scaler.transform(X2_test)

logreg2 = LogisticRegression(max_iter=2000)
logreg2.fit(X2_train_scaled, y2_train)

coef_table = pd.DataFrame({
    "Feature": ["balance", "income"],
    "Coefficient": logreg2.coef_[0],
})
print("Intercept:", round(logreg2.intercept_[0], 4))
coef_table


After scaling, a coefficient is “what happens to \(z\) when that feature goes up by **one standard deviation**.” That is why these numbers are not on the same scale as \(\beta_1\) from the one-predictor model.


**Q28.** Which coefficient is positive, and which is negative?


In [ ]:
q28 = """


"""


**Q29.** In words: if two customers have the same income, what happens to the default score \(z\) when balance is higher? What if income is higher and balance stays the same?


In [ ]:
q29 = """


"""


## Part 10 — The decision boundary

With two predictors,

$$
z = \beta_0 + \beta_1 X_1 + \beta_2 X_2
$$

At a 0.50 threshold, \(p(X)=0.50\) when \(z=0\). So the boundary is the straight line

$$
\beta_0 + \beta_1 X_1 + \beta_2 X_2 = 0
$$

Logistic regression is a **linear classifier**: the boundary is a line (or a plane in more dimensions), even though the probability curve is S-shaped.


In [ ]:
b0 = logreg2.intercept_[0]
b_bal = logreg2.coef_[0][0]
b_inc = logreg2.coef_[0][1]

x_vals = np.linspace(X2_train_scaled[:, 0].min(), X2_train_scaled[:, 0].max(), 200)
y_vals = -(b0 + b_bal * x_vals) / b_inc

no = y2_test == 0
yes = y2_test == 1

plt.scatter(X2_test_scaled[no, 0], X2_test_scaled[no, 1], alpha=0.35, s=16, label="actual No")
plt.scatter(X2_test_scaled[yes, 0], X2_test_scaled[yes, 1], alpha=0.7, s=22, label="actual Yes")
plt.plot(x_vals, y_vals, color="black", label="boundary  (z = 0, p = 0.50)")
plt.xlabel("Scaled balance")
plt.ylabel("Scaled income")
plt.title("Can you see points on the wrong side of the line?")
plt.legend()
plt.xlim(-2, 4)
plt.ylim(-3, 4)
plt.show()


**Q30.** What does a point on the **wrong side** of the line mean? Is that a false positive, a false negative, or could it be either?


In [ ]:
q30 = """


"""


**Q31.** If we kept the same fitted model but changed the threshold from 0.50 to 0.30, would this line stay in the same place? (Hint: \(p=0.30\) is not \(z=0\).)


In [ ]:
q31 = """


"""


## Part 11 — Practice from the last lecture slide

These six customers are **made up**, like the table on the last slide. You do not need Python for this part. Use only the probabilities and the thresholds.

| Customer | Balance | Prob. of default | Actual |
|---|---|---|---|
| A | \$400 | 0.05 | No |
| B | \$1,100 | 0.32 | No |
| C | \$1,450 | 0.46 | Yes |
| D | \$1,700 | 0.61 | No |
| E | \$2,100 | 0.88 | Yes |
| F | \$2,400 | 0.96 | Yes |


**Q32.** Using a threshold of **0.50**, what class would you assign to each customer A–F? Which predictions are wrong?


In [ ]:
q32 = """


"""


**Q33.** Now use a threshold of **0.30**. Who changes class? What happens to the number of false positives and false negatives? Is 0.30 necessarily better?


In [ ]:
q33 = """


"""


**Q34.** Customer C actually defaults, but the model gives 0.46. Is that probability itself necessarily unreasonable? Why or why not?


In [ ]:
q34 = """


"""


**Q35.** Which of these is **inference**, and which is **prediction**? (a) “Does balance have a significant relationship with default?” (b) “Can balance help us predict default for new customers?”


In [ ]:
q35 = """


"""


## Extra — add student status

This is optional extra practice. Student status is Yes / No, so we first turn it into 0 / 1.


In [ ]:
Default["student_yes"] = (Default["student"] == "Yes").astype(int)
Default[["student", "student_yes"]].head()


In [ ]:
X_student = Default[["student_yes"]]
logreg_student = LogisticRegression()
logreg_student.fit(X_student, y)

print("Intercept:", round(logreg_student.intercept_[0], 4))
print("Student coefficient:", round(logreg_student.coef_[0][0], 4))


**Q36.** Is the student-only coefficient positive or negative? In this simple model, are students estimated to have a higher or lower chance of default than non-students?


In [ ]:
q36 = """


"""


Now put student status in a model that also has balance and income. Scale the training features first, as in Part 9.


In [ ]:
X3 = Default[["balance", "income", "student_yes"]]
X3_train, X3_test, y3_train, y3_test = train_test_split(
    X3, y, test_size=0.25, random_state=42, stratify=y
)

scaler3 = StandardScaler()
X3_train_scaled = scaler3.fit_transform(X3_train)

logreg3 = LogisticRegression(max_iter=2000)
logreg3.fit(X3_train_scaled, y3_train)

# Scaled so balance, income, and student can be compared more fairly.
pd.DataFrame({
    "Feature": ["balance", "income", "student_yes"],
    "Coefficient": logreg3.coef_[0],
})


**Q37.** When balance and income are in the model too, does the student coefficient still have the same sign as in the student-only model? What might that tell you?


In [ ]:
q37 = """


"""


## Before you turn this in

- Every `q1`, `q2`, … box should have a short written answer, not just a blank.
- You do not need perfect sentences. A few clear lines are enough.
- If a cell failed, say so in the nearest answer box and write what you tried.
